In [ ]:
### Now that we have NPIX_BASICPROC_BHW which loads our NWB data, and MSIT_PARSING_BHW which contains ParseTrials and ConditionParsing
# We can start the MSIT_BASICPROC tranlsation..

subj_name  = 'NPIX_MSIT_004'
sig_window = 30 # ms: firing rate must stay above threshold for this long to count
q_shuffles = 100 # number of shuffles for significance testing

# same dict for parse_trials needed below for intervals
interval = {
    'pre_stim':  750,
    'post_stim': 1750,
    'pre_iti':   250,
    'post_iti':  2500,
    'pre_resp':  1250,
    'post_resp': 750
}

# Need to load stuff from NPIX_BASICPROC
# In matlab you have to dump the units_clean_df etc into the workspace..
# Here we'll just run NPIX_BASICPROC_BHW.ipynb first and it should automatically load those

### BEHAVIORAL ANALYSIS

# Find longest trial per matlab
trial_length = trials_df['trial_end_time'] - trials_df['fixaton_time'] ######## TYPO 'fixaton' is how it's spelled in NWB
##### GO BACK to check the fixation_time typo in both NPIX_BASICPROC and MSIT_PARSING! ######

longest_trial = trial_length.max()

# we can use a dict to do the behavioral.mnRT thing instead of an array
# MATLAB can't name dict keys whereas python you can
# Since not looping through by index or anything, can use a dict (each key in the dict just holds one value).
rt = trials_df['response_time'] - trials_df['stimulus_time'] # reaction time per trial

behavioral_mnRT = {
    'reaction_time': rt,
    'all_mean': rt.mean(), 'all_min': rt.min(), 'all_max': rt.max(),
    'cong_mean': rt[trials_df['Condition'] == 1].mean(), # Condition 1 = congruent
    'cong_min': rt[trials_df['Condition'] == 1].min(),
    'cong_max': rt[trials_df['Condition'] == 1].max(),
    'incong_mean': rt[trials_df['Condition'] == 2].mean(), # Condition 2 = incongruent
    'incong_min': rt[trials_df['Condition'] == 2].min(),
    'incong_max': rt[trials_df['Condition'] == 2].max(),
}

# MATLAB: if behavioral.mnRT(:,3)*1000 > interval.postStim → warning
# Check whether the longest RT fits inside our post-stim window
if behavioral_mnRT['all_max'] * 1000 > interval['post_stim']:
    print("WARNING: post-stim window is shorter than the longest reaction time")

# MATLAB: behavioral.accuracy = sum(ResponseAccuracy) / length(ResponseAccuracy)
# ResponseAccuracy is already 0/1, so mean() = proportion correct
behavioral_mnRT['accuracy_all']  = trials_df['ResponseAccuracy'].mean()
behavioral_mnRT['accuracy_cong'] = trials_df.loc[trials_df['Condition'] == 1, 'ResponseAccuracy'].mean()

# this gives us the bounds of the whole recording (min and max spike time of all units)
start_spike_time = min(spikes[0]  for spikes in unit_spikes)
final_spike_time = max(spikes[-1] for spikes in unit_spikes)

# need to find trials with no spikes
# to_numpy will convert the pandas to a plain numpy array, removing the index, so that positions of the column are just 0, 1, 2, etc
no_spike_mask = trials_df['trial_end_time'].tonumpy() > final_spike_time # outputs as boolean array
if no_spike_mask.any(): # if there are any bad trials at all..
    # argmax gives us the position of the largest value, which in the boolean is '1' i.e. 'true'
    # in this case argmax is the first "true", ie first trial with no spike
    # this will be used in parse_trials, where we say n_trials = end_trials
    # this works because the argmax output effectively counts however many trials came before it:
    # e.g., if end_trials = 3, that means 0, 1, 2 were all 'false' and 3 is the first 'true',
    # so the position of the first 'true' is equal to the count of 'false' before it!
    end_trials = int(np.argmax(no_spike_mask)) 

# just in case there are no bad trials..
else:
    end_trials = len(trials_df)

### we need to keep spike arrays > 0.5 Hz but also keep the unit that each array comes from
duration = final_spike_time - start_spike_time
n_secs   = round(duration) - 1

## sliding 1-sec window that records rate in each window (computed over ALL units, before filtering)
# we will also take average rate in 1st half and 2nd half and see if the rate drifts
# spike_rate = overall firing rate per unit (spikes / total duration); used for the 0.5 Hz filter below
spike_rate = np.array([len(spikes) / duration for spikes in unit_spikes])

spike_rate_moving = np.zeros((len(unit_spikes), n_secs)) # define array: one row per unit
for ii, spikes in enumerate(unit_spikes): # outer loop to go through units
    for jj in range(n_secs): # inner loop: each second
        window = spikes[(spikes > start_spike_time + jj) & (spikes < start_spike_time + jj + 1)] # filter to this 1-second bin
        spike_rate_moving[ii, jj] = len(window) # count spikes in that bin

## now filter: keep only units firing > 0.5 Hz
# keep = boolean mask T/F, one per unit (True where rate > 0.5)
keep = spike_rate > 0.5
# zip walks unit_spikes and keep side-by-side: (unit_spikes[0], False).. etc
# "for spikes, k in zip() if k" = give me spikes only where k is True
unit_spikes_f = [spikes for spikes, k in zip(unit_spikes, keep) if k] # 'if k' literally means 'if k is True'

# IGNORE for now but could be used later
# if I need to align metadata with the "keep" units, index the DataFrame with the same mask: unit_data[keep],
# then reset_index(drop=True) renumbers the rows from 0 so you don't keep random leftover index #s.
# unit_data_f = unit_data[keep].reset_index(drop=True)

spike_rate_first
spike_rate_second